# FTIR ML Solver v3 — Colab Training

## Setup
Go to **Runtime → Change runtime type → GPU (T4)** before running.

> ⏱ Estimate: ~2 min/epoch on T4 with 10 000 synthetic + 557 reference samples.
> For 100 epochs with 50 000 synthetic samples use the [Full Training] config below.

In [ ]:
# Clone the public repo — no login required
!git clone https://github.com/DrSuppe/FTIR_absorbtion_ML.git /content/ftir
%cd /content/ftir
!pip install -e . -q

In [ ]:
# Upload your reference spectra (not stored in the repo — binary files)
# 1. On your Mac: zip the spc_files folder:
#      zip -r spc_files.zip reference_spectra/spc_files/
# 2. Run this cell, click 'Choose Files', upload spc_files.zip
# 3. It will be unzipped automatically
import os
from google.colab import files

uploaded = files.upload()  # upload spc_files.zip here
for fname in uploaded:
    if fname.endswith('.zip'):
        os.makedirs('reference_spectra', exist_ok=True)
        os.system(f'unzip -q "{fname}" -d reference_spectra/')
        print(f'Unzipped {fname} into reference_spectra/')

# Rebuild the manifest from the newly uploaded SPC files
from ftir_analysis.manifesting import build_manifest
m = build_manifest()
print(f'Manifest ready: {len(m)} spectra, species: {sorted(m.species.unique())}')

In [ ]:
# ── Quick smoke test (~10 min on T4) ──────────────────────────────
!python3 train.py \
    --n-synthetic 5000 \
    --epochs 10 \
    --batch-size 64 \
    --log-every 2

In [ ]:
# ── Full training run (T4, ~1–2 hrs) ─────────────────────────────
# Uncomment when ready for a real training run.
# !python3 train.py \
#     --n-synthetic 50000 \
#     --epochs 100 \
#     --batch-size 128 \
#     --lr 3e-4 \
#     --warmup-epochs 5 \
#     --log-every 5

In [ ]:
# [Optional] Persist checkpoint to Google Drive so it survives session timeouts
# from google.colab import drive
# drive.mount('/content/drive')
# Then add to your train.py call:
#   --checkpoint-dir /content/drive/MyDrive/ftir_checkpoints

In [ ]:
# View the per-species MAE plot from the last epoch
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.image as mpimg

plots = sorted(Path('reports').glob('mae_per_species_*.png'))
if plots:
    img = mpimg.imread(plots[-1])
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Latest MAE plot: {plots[-1].name}')
    plt.show()
else:
    print('No MAE plots found — check training logs.')

In [ ]:
# Download the best checkpoint to your local machine
from google.colab import files
files.download('checkpoints/best_model.pt')